In [ ]:
import json
from typing import List, Dict, Any
import numpy as np

In [ ]:
def prompt_builder(prediction, top_features_info):
    if isinstance(prediction, dict):
        label = prediction.get("label")
        confidence = prediction.get("confidence")
    
    features_str = json.dumps(top_features_info, indent=2)
    
    prompt_part1 =  f"""
    You are an expert SOC Level-3 analyst specializing in Explainable Intrusion Detection Systems.
    The IDS has already finalized the detection result. Your role is NOT to classify attacks. Your task is ONLY to interpret the IDS decision, assess the operational Threat Level, recommend proportional response actions, and generate an evidence-based security report.
    ### IDS Output (Immutable)
    Treat the following values as immutable facts. Never modify, reinterpret, correct, or question them.
    Attack Label: {label}
    Confidence Score: {confidence:.4f}
    Most Influential Features: {features_str}
    ### Reasoning Policy
    Use ONLY the supplied evidence. Do not introduce new network events, feature values, indicators, or observations that are not explicitly provided.
    Never fabricate, speculate, or infer unsupported attacker behavior, compromise status, business impact, CVEs, MITRE ATT&CK techniques, or any other security artifacts.
    If the evidence is insufficient, explicitly state the uncertainty.
    Every conclusion, threat assessment, and recommendation must be explicitly supported by the supplied influential features.
    ### Threat Severity Assessment
    Threat Level represents operational response priority, NOT confirmed compromise or business impact.
    Assess Threat Level by jointly considering:
    • Attack Nature
    Use the attack label only as contextual information; never determine severity from the attack label alone.
    • IDS Confidence
    Confidence reflects only the reliability of the IDS prediction and must never be interpreted as evidence of attack severity or compensate for insufficient supporting evidence.  
    • Evidence Strength
    Determine whether the supplied influential features consistently and sufficiently support the IDS decision without major conflicting evidence.
    Do not infer evidence that is absent from the supplied feature list.
    • Observable Impact
    Estimate only the operational impact directly supported by the available evidence.
    Severity Rules
    LOW: Weak evidence or low operational priority.
    MEDIUM: Evidence supports suspicious activity requiring investigation.
    HIGH: Strong evidence supports malicious activity requiring prompt response.
    CRITICAL: Assign only when the available evidence strongly supports an event requiring immediate incident response. Never assign CRITICAL solely because of the attack category. When uncertain, choose the lower severity and explain why.
    ### Response Planning
    Recommendation Quality Requirements
    • Every recommended action MUST be explicitly supported by one or more of the supplied influential features.
    • Recommendations should prioritize validating the observed feature pattern before assuming malicious activity.
    • Whenever appropriate, explicitly reference the influential feature(s) that motivated the recommendation.
    • Do not recommend containment, blocking, isolation, or eradication solely because of the predicted attack label.
    • The urgency of recommendations MUST be proportional to the assessed Threat Level.
    • Immediate actions should reduce uncertainty or validate the alert.
    • Short-Term actions should investigate recurrence and correlate additional telemetry.
    • Long-Term actions should improve detection quality, monitoring, and response processes based on investigation outcomes.
    • Avoid generic recommendations that could apply to any IDS alert.
    • Avoid repeating substantially similar recommendations across Immediate, Short-Term, and Long-Term sections.
    • If a recommendation depends on additional telemetry or infrastructure (e.g., firewall logs, endpoint telemetry, packet capture), explicitly state "if available" unless its availability is given in the input.
    • Do not describe feature values or behaviors as high, low, short, long, abnormal, unusual, elevated, reduced, or similar qualitative terms unless such characterization is explicitly supported by a provided baseline, threshold, comparison value, or stated definition.
    ### Quality Check
    Before producing the final report, internally verify that:
    - every conclusion is evidence-supported;
    - Threat Level is justified;
    - recommendations are consistent with Threat Level;
    - no unsupported assumptions or contradictions remain.
    Return ONLY a valid JSON object.
    Attack Interpretation:
    Provide a concise explanation describing why the IDS reached its decision, explicitly relating the supplied influential features to the observed network behavior. Do not restate the attack label without explanation.
    Assessment Confidence:
    Return one of HIGH, MEDIUM, or LOW based on the strength and consistency of the available evidence.
    Security Alert:
    Provide a concise analyst-oriented summary (maximum 2–3 sentences) describing the assessed risk and the primary recommended action.
    """
    
    prompt_part2 = """
    {
    "attack_interpretation": "...",
    "threat_level": {
        "level": "...",
        "reasoning": "..."
    },
    "assessment_confidence": "...",
    "recommended_actions": {
        "immediate": [],
        "short_term": [],
        "long_term": []
    },
    "security_alert": "..."
    }
    """
    
    prompt_context = prompt_part1 + prompt_part2
    
    return prompt_context.strip()

# prompt_2 = prompt_builder(fixed_prediction, top_features_with_info)
# print(prompt_2)

In [ ]:
judge_prompt = """
You are an independent evaluator specializing in Explainable Intrusion Detection Systems.
Your task is NOT to analyze the network traffic and NOT to generate a new security report.
Your ONLY task is to evaluate the quality of an existing IDS explanation according to the three evaluation metrics defined below.
You will receive:
1. The original input prompt given to the explanation model.
2. The complete output generated by the explanation model.
Treat the IDS output contained in the input prompt (attack label, confidence score, influential features, feature definitions, and reasoning constraints) as the ONLY source of truth.
Do NOT use external cybersecurity knowledge to justify or reject statements unless they directly contradict the supplied feature definitions or immutable IDS information.
Evaluation Metric 1: Explanation Correctness
Definition:
Evaluate whether the explanation correctly interprets the supplied NetFlow features and whether every conclusion is logically supported by those features.
An explanation is CORRECT only if ALL interpretations are semantically valid.
Mark Incorrect if, for example,
• a feature is interpreted incorrectly,
• conclusions are unsupported,
• causal relationships are invented,
• the explanation misinterprets the meaning of a feature,
• conclusions contradict the supplied evidence.
Evaluation Metric 2: Feature Consistency
Definition:
Evaluate whether the explanation correctly uses the feature names, values, directions, units, definitions, and relationships exactly as provided in the input.
Check for:
• incorrect feature definitions,
• incorrect units,
• incorrect numerical interpretation,
• incorrect feature direction,
• referring to nonexistent feature values,
• confusing one feature with another,
• incorrect interpretation of feature semantics.
The explanation is Feature Consistent only if NO feature inconsistency exists.
Evaluation Metric 3: Factual Consistency
Definition:
Evaluate whether every factual statement is grounded in the supplied IDS evidence.
Hallucinations include:
• invented observations,
• fabricated attacker behavior,
• fabricated evidence,
• invented network events,
• invented indicators,
• invented feature values,
• invented security artifacts,
• unsupported assumptions,
• unsupported operational impact,
• unsupported MITRE ATT&CK techniques,
• unsupported CVEs,
• unsupported compromise claims,
• unsupported business impact.
Statements explicitly marked as uncertainty (e.g., "if available", "cannot be determined") should NOT be considered hallucinations.
Scoring Rules
For EACH metric return:
score:
1 = Pass
0 = Fail
reason:
A concise explanation (1–3 sentences).
evidence:
Quote the exact sentence(s) from the model output that justify your decision.
If no problematic sentence exists, return an empty list.
Evaluation Principles
• Evaluate ONLY the generated explanation.
• Do NOT rewrite the explanation.
• Do NOT suggest improvements.
• Do NOT generate alternative outputs.
• Do NOT penalize brevity.
• Do NOT reward writing style.
• Focus only on correctness, consistency, and factual grounding.
Return ONLY valid JSON.
{
  "explanation_correctness": {
    "score": 1,
    "reason": "...",
    "evidence": []
  },
  "feature_consistency": {
    "score": 1,
    "reason": "...",
    "evidence": []
  },
  "factual_consistency": {
    "score": 1,
    "reason": "...",
    "evidence": []
  },
  "overall_comment": "Brief overall assessment."
}
"""

In [ ]:
llm_dataset = []
for sample in clean_samples:
    prediction = sample["prediction"]
    features = sample["top_features_with_info"]
    prompt = prompt_builder(prediction, features)
    
    llm_dataset.append({
    "sample_id": sample["sample_id"],
    "prompt_version":"v1",
    "prompt":prompt
    })

In [ ]:
print(llm_dataset[0])

In [ ]:
clean_dataset = convert_numpy(
    llm_dataset
)

with open(
    "../LLM_Inputs/llm_prompts.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        clean_dataset,
        f,
        ensure_ascii=False,
        indent=4
    )

In [ ]:
import json
from openai import OpenAI
import time 

with open("../LLM_inputs/llm_prompts.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

test_prompts = prompts[:]
client = OpenAI(
    api_key= "API-Key",
    base_url="https://api.avalai.ir/v1")

In [ ]:
latency_list = []

In [ ]:
with open("./LLM_Inputs/llm_prompts.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

test_prompts = prompts[:]
print(len(test_prompts))
# print(test_prompts[0])

samples = []
for i in range(len(test_prompts)):
    start_time = time.perf_counter()
    response = client.responses.create(
        model="gpt-5.6-terra",
        input=test_prompts[i]["prompt"]
    )

    samples.append({"sample_id": test_prompts[i]["sample_id"],
                    "prompt": test_prompts[i]["prompt"],
                    "output": response.output_text})

    end_time = time.perf_counter()

    latency = end_time - start_time

    print(f"LLM Response Time: {latency:.3f} seconds")
    latency_list.extend([latency])

In [ ]:
with open(
    "../LLM_Inputs/llm_outputs_1.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        samples,
        f,
        ensure_ascii=False,
        indent=4
    )
    
# ---------- Paths ----------
INPUT_JSON = "../LLM_Inputs/llm_outputs_1.json"
OUTPUT_JSON = "../LLM_Inputs/judge_inputs_1.json"


# ---------- Load samples ----------
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    samples = json.load(f)

judge_inputs = []

for sample in samples:

    full_prompt = f"""{judge_prompt}
Original Input Prompt:
{sample["prompt"]}
Model Output:
{sample["output"]}
"""

    judge_inputs.append({
        "sample_id": sample["sample_id"],
        "judge_prompt": full_prompt
    })

# ---------- Save ----------
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(judge_inputs, f, indent=2, ensure_ascii=False)

print(f"Saved {len(judge_inputs)} judge prompts to {OUTPUT_JSON}")



with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

evaluation_data = data[:]
print(len(evaluation_data))


judge_output = []
for i in range(len(evaluation_data)):
    
    response = client.responses.create(
        model="gpt-5.5",
        input=evaluation_data[i]["judge_prompt"]
    )

    judge_output.append({"sample_id": evaluation_data[i]["sample_id"],
                         "response": response.output_text})

In [ ]:
file_path = "../results/final_results.json"

data.append(judge_output)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

In [ ]:
import json

with open("../results/final_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

criteria = [
    "explanation_correctness",
    "feature_consistency",
    "factual_consistency"
]

for item in judge_output:
    sample_id = item["sample_id"]
    response = json.loads(item["response"])

    for criterion in criteria:
        if response[criterion]["score"] == 0:
            print("=" * 80)
            print("Sample ID:", sample_id)
            print("Criterion:", criterion)
            print("Score:", response[criterion]["score"])
            print("Reason:", response[criterion]["reason"])